# Human brain MERFISH excitatory neuron figure revisions

This notebook integrates the previous Human cluster/embedding panel fixes into one read-only plotting workflow for `RealData_HumanBrainMERFISH`. Revised PNG/PDF outputs are written to `FigureReproducing/human brain`.

Purpose: support the main-text result that stGP resolves laminar architecture in human DLPFC excitatory neurons, with the main benchmark anchored on the L4/RORB marker program. Representative slices are aligned strictly by `id_region`; the 82-year representative slice is fixed to `5657_rep1` for this reproduction.

In [ ]:

from pathlib import Path
from types import SimpleNamespace
import os
import sys
import warnings

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
from matplotlib.offsetbox import AnchoredOffsetbox, HPacker, TextArea
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
from scipy.optimize import linear_sum_assignment
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings('ignore', category=FutureWarning)

BASE = Path(os.environ.get('STGP_REPRO_ROOT', Path.cwd().resolve()))
if not (BASE / 'FigureReproducing').exists():
    BASE = BASE.parent
FIG_REPRO_DIR = BASE / 'FigureReproducing'
HUMAN_DIR = BASE / 'RealData_HumanBrainMERFISH'
BENCHMARK_RESULTS_DIR = HUMAN_DIR / 'Results' / 'benchmark' / 'ext'
OUT_DIR = FIG_REPRO_DIR / 'human brain ext'
OUT_DIR.mkdir(parents=True, exist_ok=True)

STALE_OUTPUT_STEMS = [
    'clust_MEFISTO', 'clust_Popari', 'clust_STAMP', 'clust_SpatialPCA',
    'MEFISTO', 'Popari', 'STAMP', 'SpatialPCA',
    'human_marker_region_ARI', 'human_marker_region_NMI',
    'human_L4_embedding_vs_marker_correlation',
]
for stale_svg in OUT_DIR.glob('*.svg'):
    stale_svg.unlink(missing_ok=True)
for stale_stem in STALE_OUTPUT_STEMS:
    for suffix in ('.png', '.pdf'):
        (OUT_DIR / f'{stale_stem}{suffix}').unlink(missing_ok=True)

for pth in (HUMAN_DIR, FIG_REPRO_DIR):
    pth_str = str(pth)
    if pth_str not in sys.path:
        sys.path.insert(0, pth_str)
from utils import as_1d_array, best_program_by_correlation
from benchmarking_ext import (
    BASELINE_OBSM_KEYS,
    CELLTYPE2_LAYER_MAP,
    LAYER_COLORS,
    LAYER_MARKERS,
    LAYER_SPECS,
    METHODS,
    _continuous_limits,
    _representative_slices_by_age,
    _slice_ages,
    _slice_order,
)
from plots import METHOD_COLORS
from plot import (
    DPI,
    NM_W_FULL,
    NM_W_HALF,
    NM_W_SINGLE,
    draw_alpha_ci,
    ordered_gene_blocks,
    ordered_stgp_alpha,
    p_to_stars,
    save_pair as save_figure_pair,
    setup_publication_style,
)

def load_method(method, result_dir, *, celltype='ext'):
    filenames = {
        'STAMP': 'adata_with_scores.h5ad',
        'MEFISTO': 'adata_with_scores.h5ad',
        'Popari': 'res_popari.h5ad',
        'SpatialPCA': 'adata_with_scores.h5ad',
    }
    if method not in filenames:
        raise ValueError(f'Unsupported method: {method}')
    path = Path(result_dir) / filenames[method]
    return SimpleNamespace(adata=sc.read_h5ad(path))

setup_publication_style('human')


def save_pair(fig, stem, *, out_dir=OUT_DIR, dpi=DPI, bbox_inches='tight', pad_inches=0.04):
    return save_figure_pair(
        fig,
        stem,
        out_dir=out_dir,
        dpi=dpi,
        bbox_inches=bbox_inches,
        pad_inches=pad_inches,
        vector_pdf=True,
    )

print(f'Revised Human brain figures will be saved to: {OUT_DIR}')


Revised Human brain figures will be saved to: /home/byual/stGP-0529/FigureReproducing/human brain


In [ ]:

adata = sc.read_h5ad(HUMAN_DIR / 'Results' / 'stgp' / 'ext' / 'adata_with_scores.h5ad')
adata_log = sc.pp.log1p(adata.copy(), copy=True)

def _load_baselines_enabled():
    return os.environ.get('STGP_LOAD_BASELINES', '1').lower() not in {'0', 'false', 'no'}


def _load_optional_method(method, result_dir, *, celltype='ext'):
    try:
        return load_method(method, result_dir, celltype=celltype)
    except (FileNotFoundError, OSError) as exc:
        print(f'[skip] {method} baseline result is unavailable: {result_dir} ({exc})')
        return None


baseline_results = {}
if _load_baselines_enabled():
    baseline_specs = {
        'STAMP': HUMAN_DIR / 'Results' / 'baselines' / 'stamp_k=3' / 'ext',
        'MEFISTO': HUMAN_DIR / 'Results' / 'baselines' / 'mefisto' / 'ext',
        'Popari': HUMAN_DIR / 'Results' / 'baselines' / 'popari' / 'ext',
        'SpatialPCA': HUMAN_DIR / 'Results' / 'baselines' / 'spatialpca' / 'ext',
    }
    for method, result_dir in baseline_specs.items():
        loaded = _load_optional_method(method, result_dir)
        if loaded is not None:
            baseline_results[method] = loaded
else:
    print('[skip] Baseline result loading disabled by STGP_LOAD_BASELINES=0.')

baseline_adatas = {}
for method, result in baseline_results.items():
    bl = result.adata
    if bl.obs_names.equals(adata.obs_names):
        baseline_adatas[method] = bl
    elif set(bl.obs_names.astype(str)) == set(adata.obs_names.astype(str)):
        baseline_adatas[method] = bl[adata.obs_names].copy()
        print(f'Reordered {method} to match stGP obs_names.')
    else:
        print(f'[skip] {method} cells cannot be aligned to stGP by obs_names.')

slices_sorted = _slice_order(adata)
rep_slices = _representative_slices_by_age(adata, slices_sorted)
# Use the alternate 82-year section requested for this revision.
rep_slices = ['5657_rep1' if sid == '5657_rep0' else sid for sid in rep_slices]
if '5657_rep1' not in rep_slices:
    raise ValueError(f'Expected 82-year representative slice 5657_rep1, got {rep_slices}')
age_by_slice = _slice_ages(adata, slices_sorted)
xy = np.asarray(adata.obsm['spatial'])
ids = adata.obs['id_region'].astype(str).to_numpy()
obs_index = pd.Index(adata.obs_names.astype(str))

# L4/RORB is the representative excitatory-neuron benchmark in the manuscript.
layer = 'L4'
gene = LAYER_MARKERS[layer]
if adata_log is not None and gene in adata_log.var_names:
    gene_idx = np.where(adata_log.var_names == gene)[0][0]
    rorb_log = as_1d_array(adata_log.X[:, gene_idx])
else:
    rorb_log = None
    print('[skip] RORB expression-dependent panels: processed ext AnnData is unavailable or lacks RORB.')

if rorb_log is not None:
    bl_k_l4 = {
        method: best_program_by_correlation(baseline_adatas[method], BASELINE_OBSM_KEYS[method], rorb_log)
        for method in ['MEFISTO', 'Popari', 'STAMP', 'SpatialPCA']
        if method in baseline_adatas
    }
else:
    bl_k_l4 = {}
    if baseline_adatas:
        print('[skip] Baseline L4/RORB embedding panels: RORB expression is unavailable; no FigureReproducing cache is read.')

celltype2 = adata.obs['celltype2'].astype('string')
celltype2_display = pd.Series(
    celltype2.map(CELLTYPE2_LAYER_MAP).fillna('ext').astype(object).to_numpy(),
    index=obs_index,
    name='celltype2_layer_or_ext',
)
if adata_log is not None:
    marker_X = adata_log[:, list(LAYER_MARKERS.values())].X
    marker_expr = marker_X.toarray() if sp.issparse(marker_X) else np.asarray(marker_X)
    marker_z = (marker_expr - marker_expr.mean(axis=0)) / (marker_expr.std(axis=0) + 1e-8)
    marker_region = pd.Series(
        np.array(list(LAYER_MARKERS.keys()))[np.argmax(marker_z, axis=1)],
        index=obs_index,
        name='marker_region',
    )
else:
    marker_region = None

cluster_palette = {
    'L2/3': LAYER_COLORS['L2/3'],
    'L4': LAYER_COLORS['L4'],
    'L5/6': LAYER_COLORS['L5/6'],
    'ext': LAYER_COLORS.get('ext', '#BFBFBF'),
    '#unassigned': LAYER_COLORS.get('ext', '#BFBFBF'),
    'unassigned': LAYER_COLORS.get('ext', '#BFBFBF'),
}
legend_items = [('L2/3', 'L2/3'), ('L4', 'L4'), ('L5/6', 'L5/6'), ('ext', 'ext')]

summary = pd.DataFrame({
    'representative_id_region': rep_slices,
    'age_years': [age_by_slice[s] for s in rep_slices],
    'n_cells': [int((ids == s).sum()) for s in rep_slices],
})
print(summary.to_string(index=False))
print('Best L4/RORB programs, 1-based:', {m: k + 1 for m, k in bl_k_l4.items()})


In [ ]:

def clean_cluster_labels(values):
    vals = pd.Series(values).astype('string').fillna('ext').astype(str)
    return vals.replace({'#unassigned': 'ext', 'unassigned': 'ext', '<NA>': 'ext'}).to_numpy()


def one_to_one_cluster_labels(raw_pred, y_true, *, target_labels=('L2/3', 'L4', 'L5/6')):
    raw = pd.Series(raw_pred).astype('string')
    truth = pd.Series(y_true).astype('string')
    valid = raw.notna() & truth.notna()
    raw_levels = sorted(raw[valid].astype(str).unique(), key=lambda x: int(x) if x.isdigit() else x)
    true_levels = [label for label in target_labels if label in set(truth[valid].astype(str))]
    if not raw_levels or not true_levels:
        return pd.Series('ext', index=raw.index, dtype=object)

    table = pd.crosstab(raw[valid].astype(str), truth[valid].astype(str)).reindex(
        index=raw_levels, columns=true_levels, fill_value=0
    )
    row_ind, col_ind = linear_sum_assignment(-table.to_numpy())
    mapping = {raw_levels[i]: true_levels[j] for i, j in zip(row_ind, col_ind)}

    # If there are more clusters than target labels, give any leftover cluster its closest class.
    for raw_label in raw_levels:
        if raw_label not in mapping:
            mapping[raw_label] = str(table.loc[raw_label].idxmax())

    return raw.astype(str).map(mapping).fillna('ext')


def one_to_one_series_from_prediction_csv(csv_path, method, *, default='ext'):
    df = pd.read_csv(csv_path)
    df['cell'] = df['cell'].astype(str)
    sub = df[df['method'].astype(str) == method].copy()
    if sub.empty:
        return pd.Series(default, index=obs_index, dtype=object)

    pieces = []
    for _, chunk in sub.groupby('id_region', sort=False):
        mapped = one_to_one_cluster_labels(chunk['raw_pred'], chunk['ground_truth'])
        pieces.append(pd.Series(mapped.to_numpy(), index=chunk['cell'].astype(str), dtype=object))
    s = pd.concat(pieces) if pieces else pd.Series(dtype=object)
    return s.reindex(obs_index).fillna(default)


# Use one-to-one raw-cluster assignment for plotting labels. This preserves the k=3 layer
# correspondence instead of collapsing multiple raw clusters onto the same majority class.
marker_pred_csv = BENCHMARK_RESULTS_DIR / 'clustering' / 'marker_region' / 'cell_predictions.csv'
cell_pred_csv = BENCHMARK_RESULTS_DIR / 'clustering' / 'celltype2' / 'cell_predictions.csv'
cluster_values = {}
if marker_pred_csv.exists() and cell_pred_csv.exists():
    for method in METHODS:
        marker_series = one_to_one_series_from_prediction_csv(marker_pred_csv, method, default='ext')
        cell_series = one_to_one_series_from_prediction_csv(cell_pred_csv, method, default=np.nan)
        combined = marker_series.copy()
        combined.loc[cell_series.dropna().index] = cell_series.dropna()
        cluster_values[method] = clean_cluster_labels(combined)
else:
    print(f'[skip] Benchmark cell-prediction CSVs are unavailable under {BENCHMARK_RESULTS_DIR}.')

celltype_values = clean_cluster_labels(celltype2_display)


def add_age_label(ax, sid, *, fontsize=7.0):
    ax.text(
        0.5, 1.01, f'{age_by_slice[sid]:.0f} yr',
        ha='center', va='bottom', transform=ax.transAxes, fontsize=fontsize,
    )


def plot_cluster_tiles(values, stem, *, show_legend=False, title=None, dot_size=2.3):
    labels = clean_cluster_labels(values)
    fig, axes = plt.subplots(2, 2, figsize=(3.55, 3.58 if title else 3.42), squeeze=False)
    fig.subplots_adjust(
        left=0.02, right=0.98 if not show_legend else 0.76,
        top=0.90 if title else 0.96, bottom=0.02,
        wspace=0.02, hspace=0.12,
    )
    for ax in axes.ravel():
        ax.axis('off')
    for ax, sid in zip(axes.ravel(), rep_slices):
        mask = ids == sid
        ax.scatter(
            xy[mask, 0], xy[mask, 1],
            c=[cluster_palette.get(v, '#888888') for v in labels[mask]],
            s=dot_size, linewidths=0, rasterized=True,
        )
        ax.set_aspect('equal')
    if title:
        fig.suptitle(title, fontsize=14, fontweight='bold', y=0.985)
    if show_legend:
        handles = [Patch(facecolor=cluster_palette[key], edgecolor='none', label=label) for key, label in legend_items]
        fig.legend(handles=handles, loc='center left', bbox_to_anchor=(0.78, 0.50), frameon=False,
                   fontsize=11.5, handlelength=1.0, handletextpad=0.45)
    save_pair(fig, stem, pad_inches=0.01)


def embedding_values(method):
    key = BASELINE_OBSM_KEYS[method]
    return np.asarray(baseline_adatas[method].obsm[key])[:, bl_k_l4[method]]


def plot_embedding_tiles(method, stem=None, *, dot_size=2.3):
    vals = np.asarray(embedding_values(method), dtype=float)
    signed = method in {'MEFISTO', 'SpatialPCA'} or np.nanmin(vals) < 0
    vmin, vmax = _continuous_limits(vals, symmetric=signed)
    cmap = 'RdBu_r' if signed else 'YlOrBr'
    fig, axes = plt.subplots(2, 2, figsize=(3.70, 3.42), squeeze=False)
    fig.subplots_adjust(left=0.02, right=0.84, top=0.96, bottom=0.02, wspace=0.02, hspace=0.12)
    for ax in axes.ravel():
        ax.axis('off')
    for ax, sid in zip(axes.ravel(), rep_slices):
        mask = ids == sid
        ax.scatter(
            xy[mask, 0], xy[mask, 1],
            c=vals[mask], cmap=cmap, vmin=vmin, vmax=vmax,
            s=dot_size, linewidths=0, rasterized=True,
        )
        ax.set_aspect('equal')
    cbar_ax = fig.add_axes([0.875, 0.18, 0.030, 0.64])
    sm = ScalarMappable(norm=Normalize(vmin=vmin, vmax=vmax), cmap=cmap)
    cb = fig.colorbar(sm, cax=cbar_ax)
    cb.set_label('score', fontsize=10, labelpad=4)
    cb.ax.tick_params(labelsize=9, length=2.5, width=0.8)
    save_pair(fig, stem or method, pad_inches=0.01)


def plot_rorb_expression():
    vals = np.asarray(rorb_log, dtype=float)
    finite = vals[np.isfinite(vals)]
    vmin = 0.0
    vmax = 4.0
    fig, axes = plt.subplots(2, 2, figsize=(3.70, 3.58), squeeze=False)
    fig.subplots_adjust(left=0.02, right=0.84, top=0.90, bottom=0.02, wspace=0.02, hspace=0.12)
    for ax in axes.ravel():
        ax.axis('off')
    for ax, sid in zip(axes.ravel(), rep_slices):
        mask = ids == sid
        ax.scatter(
            xy[mask, 0], xy[mask, 1],
            c=vals[mask], cmap='YlOrBr', vmin=vmin, vmax=vmax,
            s=2.3, linewidths=0, rasterized=True,
        )
        ax.set_aspect('equal')
    cbar_ax = fig.add_axes([0.875, 0.18, 0.030, 0.64])
    sm = ScalarMappable(norm=Normalize(vmin=vmin, vmax=vmax), cmap='YlOrBr')
    cb = fig.colorbar(sm, cax=cbar_ax, ticks=[0, 2, 4])
    cb.set_label('log1p expression', fontsize=10, labelpad=4)
    cb.ax.tick_params(labelsize=9, length=2.5, width=0.8)
    fig.suptitle('RORB', fontsize=14, fontweight='bold', fontstyle='italic', fontfamily='Arial', y=0.985)
    save_pair(fig, 'RORB_expression', pad_inches=0.01)


def plot_stgp_l4_embedding_tiles():
    vals = np.asarray(adata.obsm['X_stgp_spatial'])[:, 2].astype(float)
    vmin, vmax = _continuous_limits(vals, symmetric=True)
    fig, axes = plt.subplots(2, 2, figsize=(3.70, 3.58), squeeze=False)
    fig.subplots_adjust(left=0.02, right=0.84, top=0.90, bottom=0.02, wspace=0.02, hspace=0.12)
    for ax in axes.ravel():
        ax.axis('off')
    for ax, sid in zip(axes.ravel(), rep_slices):
        mask = ids == sid
        ax.scatter(
            xy[mask, 0], xy[mask, 1],
            c=vals[mask], cmap='RdBu_r', vmin=vmin, vmax=vmax,
            s=2.3, linewidths=0, rasterized=True,
        )
        ax.set_aspect('equal')
    cbar_ax = fig.add_axes([0.875, 0.18, 0.030, 0.64])
    sm = ScalarMappable(norm=Normalize(vmin=vmin, vmax=vmax), cmap='RdBu_r')
    cb = fig.colorbar(sm, cax=cbar_ax)
    cb.set_label('score', fontsize=10, labelpad=4)
    cb.ax.tick_params(labelsize=9, length=2.5, width=0.8)
    fig.suptitle('stGP', fontsize=14, fontweight='bold', y=0.985)
    save_pair(fig, 'stGP_b', pad_inches=0.01)


In [ ]:

def plot_cluster_cell(fig, subspec, labels, *, dot_size=1.8):
    labels = clean_cluster_labels(labels)
    inner = GridSpecFromSubplotSpec(2, 2, subplot_spec=subspec, wspace=0.025, hspace=0.045)
    for i, sid in enumerate(rep_slices):
        ax = fig.add_subplot(inner[i // 2, i % 2])
        mask = ids == sid
        ax.scatter(
            xy[mask, 0], xy[mask, 1],
            c=[cluster_palette.get(v, '#888888') for v in labels[mask]],
            s=dot_size, linewidths=0, rasterized=True,
        )
        ax.set_aspect('equal')
        ax.axis('off')


def plot_embedding_cell(fig, subspec, method, *, dot_size=1.8):
    vals = np.asarray(embedding_values(method), dtype=float)
    signed = method in {'MEFISTO', 'SpatialPCA'} or np.nanmin(vals) < 0
    vmin, vmax = _continuous_limits(vals, symmetric=signed)
    cmap = 'RdBu_r' if signed else 'YlOrBr'
    inner = GridSpecFromSubplotSpec(2, 3, subplot_spec=subspec, width_ratios=[1, 1, 0.08], wspace=0.045, hspace=0.045)
    for i, sid in enumerate(rep_slices):
        ax = fig.add_subplot(inner[i // 2, i % 2])
        mask = ids == sid
        ax.scatter(
            xy[mask, 0], xy[mask, 1],
            c=vals[mask], cmap=cmap, vmin=vmin, vmax=vmax,
            s=dot_size, linewidths=0, rasterized=True,
        )
        ax.set_aspect('equal')
        ax.axis('off')
    cax = fig.add_subplot(inner[:, 2])
    sm = ScalarMappable(norm=Normalize(vmin=vmin, vmax=vmax), cmap=cmap)
    cb = fig.colorbar(sm, cax=cax)
    cb.set_label('score', fontsize=8.5, labelpad=2)
    cb.ax.tick_params(labelsize=7.5, length=2.0, width=0.7)


def plot_human_cluster_embedding_composite():
    method_order = ['SpatialPCA', 'Popari', 'MEFISTO', 'STAMP']
    available = [
        method for method in method_order
        if method in baseline_adatas and method in bl_k_l4 and method in cluster_values
    ]
    if not available:
        print('[skip] Human baseline cluster/embedding composite: no complete baseline inputs were available.')
        return

    fig = plt.figure(figsize=(7.9, 2.98 * len(available)), constrained_layout=False)
    outer = GridSpec(
        len(available), 2, figure=fig,
        left=0.055, right=0.965, top=0.965, bottom=0.085,
        width_ratios=[1.0, 1.10], hspace=0.26, wspace=0.11,
    )
    for row_i, method in enumerate(available):
        plot_cluster_cell(fig, outer[row_i, 0], cluster_values[method])
        plot_embedding_cell(fig, outer[row_i, 1], method)
        left = outer[row_i, 0].get_position(fig)
        right = outer[row_i, 1].get_position(fig)
        fig.text(
            (left.x0 + right.x1) / 2, max(left.y1, right.y1) + 0.014,
            method, ha='center', va='bottom', fontsize=13.5, fontweight='bold',
        )
    handles = [Patch(facecolor=cluster_palette[key], edgecolor='none', label=label) for key, label in legend_items]
    fig.legend(handles=handles, loc='lower center', ncol=4, frameon=False,
               bbox_to_anchor=(0.5, 0.020), fontsize=12.5, handlelength=1.0,
               handletextpad=0.45, columnspacing=1.3)
    save_pair(fig, 'human_cluster_embedding_4x2_panel', pad_inches=0.015)


In [ ]:

def stars(p):
    return p_to_stars(p, nan_label='NA', nonsig_label='ns')


def metric_boxplot(ax, df, metric, y_label, *, methods=METHODS, fontsize=11, add_brackets=True):
    data = [df.loc[df['method'] == method, metric].dropna().to_numpy(float) for method in methods]
    colors = [METHOD_COLORS.get(method, '#999999') for method in methods]
    bp = ax.boxplot(
        data, patch_artist=True, widths=0.55,
        medianprops=dict(color='black', linewidth=1.3),
        whiskerprops=dict(linewidth=0.9), capprops=dict(linewidth=0.9),
        flierprops=dict(marker='o', markersize=3.0, alpha=0.45, markeredgewidth=0),
    )
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.78)
    for flier, color in zip(bp['fliers'], colors):
        flier.set_markerfacecolor(color)
        flier.set_markeredgecolor(color)
    ax.set_xticks(range(1, len(methods) + 1))
    ax.set_xticklabels(methods, rotation=30, ha='right', fontsize=fontsize)
    if y_label:
        ax.set_ylabel(y_label, fontsize=fontsize + 2)
    ax.tick_params(axis='y', labelsize=fontsize, length=3.2, width=0.8)
    ax.grid(axis='y', linestyle='--', linewidth=0.55, alpha=0.35)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    if not add_brackets or 'stGP' not in methods:
        return
    ref = df.loc[df['method'] == 'stGP', ['id_region', metric]].rename(columns={metric: 'ref'})
    comps, raw_p = [], []
    for method in methods:
        if method == 'stGP':
            continue
        other = df.loc[df['method'] == method, ['id_region', metric]].rename(columns={metric: 'other'})
        paired = ref.merge(other, on='id_region').dropna()
        diff = paired['ref'].to_numpy(float) - paired['other'].to_numpy(float)
        if len(diff) < 2:
            p = np.nan
        elif np.allclose(diff, 0):
            p = 1.0
        else:
            p = float(wilcoxon(paired['ref'], paired['other'], alternative='greater').pvalue)
        comps.append(method)
        raw_p.append(p)
    raw_p = np.asarray(raw_p, dtype=float)
    adj = np.full_like(raw_p, np.nan)
    valid = np.isfinite(raw_p)
    if valid.any():
        _, adj[valid], _, _ = multipletests(raw_p[valid], method='holm')
    vals = np.concatenate([d[np.isfinite(d)] for d in data if np.isfinite(d).any()])
    if vals.size == 0:
        return
    ymin, ymax = float(np.nanmin(vals)), float(np.nanmax(vals))
    yr = ymax - ymin if ymax > ymin else 1.0
    y0 = ymax + 0.08 * yr
    step = 0.12 * yr
    dy = 0.03 * yr
    for i, (method, p_adj) in enumerate(zip(comps, adj)):
        x1, x2 = 1, methods.index(method) + 1
        y = y0 + i * step
        ax.plot([x1, x1, x2, x2], [y, y + dy, y + dy, y], lw=0.85, color='black', clip_on=False)
        ax.text((x1 + x2) / 2, y + dy, stars(p_adj), ha='center', va='bottom', fontsize=fontsize - 1)
    ax.set_ylim(top=y0 + len(comps) * step + 0.10 * yr)


def save_metric_figure(gt_name, metric, label, stem):
    csv_path = BENCHMARK_RESULTS_DIR / 'clustering' / gt_name / 'slice_metrics.csv'
    if not csv_path.exists():
        print(f'[skip] {stem}: missing benchmark metrics CSV {csv_path}.')
        return
    df = pd.read_csv(csv_path)
    fig, ax = plt.subplots(figsize=(4.0, 3.75), constrained_layout=True)
    metric_boxplot(ax, df, metric, label, fontsize=11)
    save_pair(fig, stem)


def plot_marker_region_ari_nmi_combined():
    csv_path = BENCHMARK_RESULTS_DIR / 'clustering' / 'marker_region' / 'slice_metrics.csv'
    if not csv_path.exists():
        print(f'[skip] human_marker_region_ARI_NMI_1x2: missing benchmark metrics CSV {csv_path}.')
        return
    df = pd.read_csv(csv_path)
    fig, axes = plt.subplots(1, 2, figsize=(7.8, 3.85), constrained_layout=True)
    metric_boxplot(axes[0], df, 'raw_ari', '', fontsize=12)
    metric_boxplot(axes[1], df, 'raw_nmi', '', fontsize=12)
    axes[0].set_title('ARI', fontsize=18, pad=8)
    axes[1].set_title('NMI', fontsize=18, pad=8)
    save_pair(fig, 'human_marker_region_ARI_NMI_1x2')


def add_layer_marker_title(ax, layer_name, marker_gene):
    normal_left = TextArea(f'{layer_name} (', textprops={'fontsize': 15, 'fontfamily': 'Arial'})
    gene_text = TextArea(marker_gene, textprops={'fontsize': 15, 'fontfamily': 'Arial', 'fontstyle': 'italic'})
    normal_right = TextArea(')', textprops={'fontsize': 15, 'fontfamily': 'Arial'})
    title_box = HPacker(children=[normal_left, gene_text, normal_right], align='center', pad=0, sep=0)
    anchored = AnchoredOffsetbox(
        loc='upper center', child=title_box, frameon=False, pad=0, borderpad=0,
        bbox_to_anchor=(0.5, 1.12), bbox_transform=ax.transAxes,
    )
    ax.add_artist(anchored)


def plot_correlation_boxplots():
    csv_path = BENCHMARK_RESULTS_DIR / 'summary' / 'source_data' / 'marker_embedding_correlations.csv'
    if not csv_path.exists():
        print(f'[skip] human_embedding_vs_marker_correlation_all_layers: missing correlation CSV {csv_path}.')
        return
    corr = pd.read_csv(csv_path)
    method_order = METHODS
    fig, axes = plt.subplots(1, 3, figsize=(10.2, 3.95), constrained_layout=True)
    for ax, layer_name in zip(axes, ['L2/3', 'L4', 'L5/6']):
        sub = corr[corr['layer'] == layer_name].copy().rename(columns={'correlation': 'corr'})
        metric_boxplot(ax, sub, 'corr', 'Pearson correlation' if ax is axes[0] else '', methods=method_order, fontsize=11)
        add_layer_marker_title(ax, layer_name, LAYER_MARKERS[layer_name])
    save_pair(fig, 'human_embedding_vs_marker_correlation_all_layers')


In [ ]:

def plot_alpha_stgp3_large():
    info = adata.uns.get('stgp', {})
    alpha = np.asarray(info.get('alpha', []), dtype=float)
    if alpha.ndim != 2 or alpha.shape[0] < 3:
        raise ValueError('stGP alpha matrix does not contain stGP3.')
    ages_ord, y, lo, hi, _ = ordered_stgp_alpha(info, 2)
    fig, ax = plt.subplots(figsize=(6.5, 5.05), constrained_layout=True)
    draw_alpha_ci(ax, ages_ord, y, lo, hi, color='#2C7FB8', scatter_s=72)
    ax.set_xlabel('Age (yr)', fontsize=24)
    ax.set_ylabel(r'Aging effect $\alpha$', fontsize=24)
    ax.tick_params(axis='both', labelsize=20, length=4.5, width=1.1)
    ax.legend(fontsize=18, loc='best')
    save_pair(fig, 'alpha_trajectory_stGP3')


def plot_w_heatmap_vertical_grouped():
    W = pd.read_csv(HUMAN_DIR / 'Results' / 'stgp' / 'ext' / 'W.csv', index_col=0)
    block_df, W_ord = ordered_gene_blocks(W, top_n_per_program=15)
    mat = W_ord.T
    vmax = 0.20
    cmap = LinearSegmentedColormap.from_list('wload_human', ['#FFFFFF', '#F6B5B8', '#B2182B'])
    fig_h = max(15.2, 0.36 * len(mat.index) + 2.4)
    fig, ax = plt.subplots(figsize=(5.25, fig_h), constrained_layout=False)
    fig.subplots_adjust(left=0.31, right=0.82, top=0.985, bottom=0.070)
    im = ax.imshow(mat.to_numpy(float), aspect='auto', cmap=cmap, vmin=0, vmax=vmax)
    ax.set_xticks(np.arange(mat.shape[1]))
    ax.set_xticklabels(mat.columns.astype(str), rotation=90, fontsize=15)
    ax.set_yticks(np.arange(mat.shape[0]))
    ax.set_yticklabels(mat.index.astype(str), fontsize=12.5, fontstyle='italic')
    ax.set_xlabel('Program', fontsize=18, labelpad=8)
    ax.set_ylabel('Top loading genes', fontsize=18, labelpad=14)
    ax.tick_params(length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)
    # Thin horizontal dividers mark the program-anchored gene blocks.
    counts = block_df.groupby('program', sort=False).size().reindex(mat.columns, fill_value=0)
    y = 0
    for count in counts:
        y += int(count)
        if y < mat.shape[0]:
            ax.axhline(y - 0.5, color='white', lw=1.2)
    cax = fig.add_axes([0.855, 0.20, 0.040, 0.60])
    cbar = fig.colorbar(im, cax=cax, ticks=[0, 0.05, 0.10, 0.15, 0.20])
    cbar.ax.set_yticklabels(['0', '0.05', '0.10', '0.15', '0.20'])
    cbar.set_label('Gene weight W', fontsize=16, labelpad=10)
    cbar.ax.tick_params(labelsize=13, length=4)
    save_pair(fig, 'W_heatmap_vertical', pad_inches=0.03)


In [ ]:

# Representative panels and composite benchmark panel.
plot_cluster_tiles(celltype_values, 'celltype2', show_legend=False, title='celltype')
if 'stGP' in cluster_values:
    plot_cluster_tiles(cluster_values['stGP'], 'stGP', show_legend=False, title='stGP')
else:
    print('[skip] stGP cluster tile: benchmark cell-prediction CSVs are unavailable.')
plot_rorb_expression()
plot_stgp_l4_embedding_tiles()
plot_human_cluster_embedding_composite()

# Metrics and correlation figures copied/redrawn from id_region-keyed source data.
plot_marker_region_ari_nmi_combined()
plot_correlation_boxplots()

# stGP3 alpha trajectory and grouped W heatmap.
plot_alpha_stgp3_large()
plot_w_heatmap_vertical_grouped()

print('Done. Output files:', len(list(OUT_DIR.glob('*.png'))), 'PNG and', len(list(OUT_DIR.glob('*.pdf'))), 'PDF')
